<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/Phase3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from google.colab import files

print("Loading 103MB Matrix...")
df = pd.read_csv("nalbari_h3_weather_matrix.csv.gz")

# 1. Ensure absolute chronological integrity per spatial zone
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(by=['hexagon', 'time']).reset_index(drop=True)

print("Matrix loaded. Initiating Phase 3 Biological State calculations...")

# ---------------------------------------------------------
# ENGINE 1: The Hydrology Mass-Balance (Leaky Capacitor)
# ---------------------------------------------------------
infiltration_rate = 5.0

# THE FIX: Explicitly assign the intermediate math to a named column in the DataFrame
df['net_water_change'] = df['precip'] - df['evapotranspiration'] - infiltration_rate

# Custom vectorized rolling calculation bounded at 0
def calculate_standing_water(series):
    water = np.zeros(len(series))
    current_w = 0
    for i, val in enumerate(series):
        current_w = max(0, current_w + val)
        water[i] = current_w
    return water

print("Running Hydrology Engine across 2,671 vector zones...")
# Now we reference the explicitly named string 'net_water_change'
df['net_standing_water'] = df.groupby('hexagon')['net_water_change'].transform(calculate_standing_water)

# ---------------------------------------------------------
# ENGINE 2: The Thermal Integration (VIDD)
# ---------------------------------------------------------
print("Running Viral Thermodynamic Engine...")

# Calculate daily thermal fuel (bounds: 14C to 34C)
df['daily_vidd'] = np.where(
    df['temp_mean'] > 34.0, 0,  # Thermal stress limit
    np.maximum(0, df['temp_mean'] - 14.0) # Accumulation
)

# 14-day rolling window representing total cohort maturity pressure
df['rolling_14d_vidd'] = df.groupby('hexagon')['daily_vidd'].transform(lambda x: x.rolling(window=14, min_periods=1).sum())

# ---------------------------------------------------------
# EXPORT
# ---------------------------------------------------------
df_bio = df[['time', 'hexagon', 'net_standing_water', 'rolling_14d_vidd']]

output_file = "nalbari_phase3_biological_state.csv.gz"
print("Compressing Final Phase 3 Matrix...")
df_bio.to_csv(output_file, index=False, compression="gzip")

print(f"Phase 3 Complete. Triggering local download for {output_file}...")
files.download(output_file)

print("\n--- SAMPLE OUTPUT ---")
print(df_bio.head(10))

Loading 103MB Matrix...
Matrix loaded. Initiating Phase 3 Biological State calculations...
Running Hydrology Engine across 2,671 vector zones...
Running Viral Thermodynamic Engine...
Compressing Final Phase 3 Matrix...
Phase 3 Complete. Triggering local download for nalbari_phase3_biological_state.csv.gz...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SAMPLE OUTPUT ---
        time          hexagon  net_standing_water  rolling_14d_vidd
0 2020-12-31  883ce03401fffff                 0.0          3.532710
1 2021-01-01  883ce03401fffff                 0.0          6.956917
2 2021-01-02  883ce03401fffff                 0.0         10.532117
3 2021-01-03  883ce03401fffff                 0.0         14.420335
4 2021-01-04  883ce03401fffff                 0.0         19.003127
5 2021-01-05  883ce03401fffff                 0.0         23.347489
6 2021-01-06  883ce03401fffff                 0.0         29.263188
7 2021-01-07  883ce03401fffff                 0.0         35.046661
8 2021-01-08  883ce03401fffff                 0.0         40.644736
9 2021-01-09  883ce03401fffff                 0.0         46.100118
